File: HeydCNN_Reimp_EvalModel-Dom
Author: James Jolly
Date: Aug 6, 2025

Purpose:
    This notebook is a quick record of evaluating dominant only GT trained Heydarian et al. CNN-LSTM benchmark
    

In [1]:
import sys
import math
import numpy as np # v1.19.5
import tensorflow as tf # version 2.2.0
from tensorflow import keras


from io import UnsupportedOperation
import pickle as pkl

# Add shared location for auxillary functions
sys.path.insert(1, './../AuxillaryFunctions')

from GenerateClassDataFuncs import GenerateEvalData_OREBA
from GenerateClassDataFuncs import GenerateEvalData_ClemCafe


from EvaluationMethodFuncs import DongEval_PtVsPt
from EvaluationMethodFuncs import KyritEval_PtVsWindow
from EvaluationMethodFuncs import TimePoint2Window
from EvaluationMethodFuncs import Window2TimePoint
from EvaluationMethodFuncs import Duration_WindowVsWindow
from EvaluationMethodFuncs import PrintStats_SingleLine


print("Finished importing libraries and functions.")

2025-08-08 09:20:52.983680: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /software/slurm/spackages/linux-rocky8-x86_64/gcc-12.2.0/anaconda3-2023.09-0-3mhml42fa64byxqyd5fig5tbih625dp2/lib
2025-08-08 09:20:52.983714: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


Finished importing libraries and functions.


In [2]:
### DEFINE CONSTANTS ###

DEBUG_PRINTS = 0

OptionSelect = 10

if OptionSelect == 1:
	MODEL_FILE = "DomClem_HeydCNN_0/models/fold1"
	FOLD_INDEX = 1 # Which index to leave out for validation
	DATABASE_FLAG = 5 # 5 = DomClem, 6 = DomOHO
elif OptionSelect == 2:
	MODEL_FILE = "DomClem_HeydCNN_0/models/fold2"
	FOLD_INDEX = 2
	DATABASE_FLAG = 5
elif OptionSelect == 3:
	MODEL_FILE = "DomClem_HeydCNN_0/models/fold3"
	FOLD_INDEX = 3
	DATABASE_FLAG = 5
elif OptionSelect == 4:
	MODEL_FILE = "DomClem_HeydCNN_0/models/fold4"
	FOLD_INDEX = 4
	DATABASE_FLAG = 5
elif OptionSelect == 5:
	MODEL_FILE = "DomClem_HeydCNN_0/models/fold0"
	FOLD_INDEX = 0
	DATABASE_FLAG = 5
elif OptionSelect == 6:
	MODEL_FILE = "DomOHO_HeydCNN_0/models/fold0"
	FOLD_INDEX = 0
	DATABASE_FLAG = 6
elif OptionSelect == 7:
	MODEL_FILE = "DomOHO_HeydCNN_0/models/fold1"
	FOLD_INDEX = 1 # Which index to leave out for validation
	DATABASE_FLAG = 6 # 5 = DomClem, 6 = DomOHO
elif OptionSelect == 8:
	MODEL_FILE = "DomOHO_HeydCNN_0/models/fold2"
	FOLD_INDEX = 2
	DATABASE_FLAG = 6
elif OptionSelect == 9:
	MODEL_FILE = "DomOHO_HeydCNN_0/models/fold3"
	FOLD_INDEX = 3
	DATABASE_FLAG = 6
elif OptionSelect == 10:
	MODEL_FILE = "DomOHO_HeydCNN_0/models/fold4"
	FOLD_INDEX = 4
	DATABASE_FLAG = 6
elif False: # Original Eval Code
	MODEL_FILE = sys.argv[1]
	FOLD_INDEX = int(sys.argv[2]) # Which index to leave out for validation
	FOLDS_TOTAL = int(sys.argv[3]) # total number of folds to split the data into
	DATABASE_FLAG = int(sys.argv[4]) # Must be 1 (Dom Hand Only) or 2 (Both Hands)
# end


FOLDS_TOTAL = 5 # total number of folds to split the data into


DATA_FREQ = 64 # Heyd Model Always at 64 Hz
CUT_sec = 2 # window is always 128 (2 x 64 hz)
STRIDE_sec = float(8.0/64.0) 
STRIDE = STRIDE_sec * DATA_FREQ

RR_FLAG = 1 # 1 if using striped fold division, 0 if using block fold division

DOWN_SAMPLE_FLAG = 0 # 1 = Down Sample data, 0 = use raw data
DOWN_SAMPLE_RATE = 1 # Num of points to consolidate into a single point
DOWN_SAMPLE_OFFSET = 0 # Offset in DS data; in range [0, DS_Rate) 


# Database Selection
if DATABASE_FLAG == 1:
	DATABASE_FILEPATH = './../Pickle_Databases/OREBA.pkl'
	RESAMPLE_FLAG_FREQ = 0 # zero if keeping original data frequency
elif DATABASE_FLAG == 2:
	DATABASE_FILEPATH = './../Pickle_Databases/OREBA_TwoHands.pkl'
	RESAMPLE_FLAG_FREQ = 0 # zero if keeping original data frequency
elif DATABASE_FLAG == 3:
	DATABASE_FILEPATH = "./../Pickle_Databases/ClemCafe.pkl"
	RESAMPLE_FLAG_FREQ = 64 # new freq since ClemCafe = 15Hz
elif DATABASE_FLAG == 4:
	DATABASE_FILEPATH = './../Pickle_Databases/CleanClemCafe.pkl'
	RESAMPLE_FLAG_FREQ = 64 # new freq since ClemCafe = 15Hz
elif DATABASE_FLAG == 5:
	DATABASE_FILEPATH = './../Pickle_Databases/ClemDomPkl.pkl'
	RESAMPLE_FLAG_FREQ = 64 # new freq since ClemCafe = 15Hz
elif DATABASE_FLAG == 6:
	DATABASE_FILEPATH = './../Pickle_Databases/OHO_Dom_v2.pkl'
	RESAMPLE_FLAG_FREQ=0 # zero if keeping original data frequency
else:
	print("!!! WARNING: INVALID DATABASE SELECTION FLAG VALUE OF {} !!!".format(DATABASE_FLAG))
	print("Expects value of:")
	print("\t1 = OneHand OREBA")
	print("\t2 = TwoHand OREBA")
	print("\t3 = ClemCafe Data")
	print("Defaulting to use of ClemCafeData...")
	DATABASE_FILEPATH = "./../Pickle_Databases/ClemCafe.pkl"
#end of Database switch statement

print("Finished Defining Constants.")

Finished Defining Constants.


In [3]:
### SELECT EVAL METHOD ###

WIN_TOLERANCE = 99
EVAL_METHOD_FLAG = 1
	# 1 = Dong Eval (Pt vs Pt)
	# 2 = Kyritsis (Pt vs Window)
	# 3 = Duration (Window vs Window)
	
GT_WINDOW_FLAG = 1 # Always grab GT Window so detections only need to run once to test both eval methods
# if EVAL_METHOD_FLAG == 2 or EVAL_METHOD_FLAG == 3:
# 	GT_WINDOW_FLAG = 1 #mark that we want Windows from GT if using a window metric
# else:
# 	GT_WINDOW_FLAG = 0 # mark that we want time points for GT metrics
# # end of Window vs Point GT Flag


# # # # # # KNOBS TO TURN FOR EVAL METHOD # # # # # #
DETECTION_METHOD_FLAG = 1 
	# Used to determine how Local Maximums are determined
	# 1 = Immediate Local Maximum; just considers points around it;
	#       Only triggers once every bite_length_sec
	# 2 = Searches each non-floored segment to find the 
	#       maximum and places the detection there; see segment for additional knobs



# # # # # Use model Predictions to place bite in Window # # # # #
# WIN_TOL now defined as user input [7] 
# WIN_TOLERANCE = 1.00 # seconds leeway to give matching windows
BiteDetectThresh=0.65 # Value used by Kyritsis in Paper to trigger a detection
bite_length_sec = 2 # duration in sec to mark a triggered detection as 
					# a bite and ignore any value in that window
delayed_offset_sec = 0.0
# add a small offset to center the predictions in window

print("Finished Setting Eval Knobs.")

Finished Setting Eval Knobs.


In [4]:
if DEBUG_PRINTS == 1:
	EVAL_METHOD_NAMES = ["Names for Eval methods (valid flags start at 1)",
			 "Dong_PtVsPt",
			 "Kyritsis_PtVsWind",
			 "Duration_WindVsWind"]

	print('CUT_sec Input is ',CUT_sec)
	print('STRIDE_sec input is ',STRIDE_sec)
	print('Training Data coming from file ',DATABASE_FILEPATH)
	print('RR_Flag Input is ',RR_FLAG)
	print('Fold_Index Input is ',FOLD_INDEX)
	print('Folds_Total Input is ',FOLDS_TOTAL)
	print('Resample Rate Input is ',RESAMPLE_FLAG_FREQ)
	print('Eval Method is ',EVAL_METHOD_NAMES[EVAL_METHOD_FLAG])
	print('WIN_TOLERANCE (if needed) is ',WIN_TOLERANCE)
	print('Offset_sec (if needed) is ',delayed_offset_sec)
	print("python setup complete")
# end of DEBUG_PRINTS

In [5]:
# # # # # Read in file raw data # # # # #
if DATABASE_FLAG == 1 or DATABASE_FLAG == 2 or DATABASE_FLAG == 6:
	compiled_eval_data = GenerateEvalData_OREBA(
			int(round(CUT_sec*DATA_FREQ)), int(round(STRIDE_sec*DATA_FREQ)),
			DATABASE_FILEPATH,
			FOLD_INDEX,	FOLDS_TOTAL, FOLD_SPLIT=RR_FLAG,
			RESAMPLE_FLAG=RESAMPLE_FLAG_FREQ,
			GT_WINDOW_FLAG = GT_WINDOW_FLAG # always grab window
			)
	pt_gt_data=[]
	for meal_num in range(len(compiled_eval_data)):
		pt_gt_data.append(Window2TimePoint(np.array(compiled_eval_data[meal_num][3])))
	print("Using OREBA Data...") 
	# NOTE: Pass GT_WINDOW_FLAG into OREBA because the original GT is given in Windows
elif DATABASE_FLAG == 3 or DATABASE_FLAG == 4 or DATABASE_FLAG == 5:
	compiled_eval_data = GenerateEvalData_ClemCafe(
			int(round(CUT_sec*DATA_FREQ)), int(round(STRIDE_sec*DATA_FREQ)),
			DATABASE_FILEPATH,
			FOLD_INDEX, FOLDS_TOTAL, FOLD_SPLIT=RR_FLAG,
			RESAMPLE_FLAG=RESAMPLE_FLAG_FREQ # Frequency of Data Collection in [Hz]
			#, RESAMPLE_FLAG=0, SMOOTHING=0 # optional flags not currently used for Paper Experiment
			)
	# NOTE: Pass GT_WINDOW_FLAG into OREBA because the original GT is given in Windows
	if GT_WINDOW_FLAG == 1:
		# Save Point GT as separate variable
		pt_gt_data=[]
		for meal_num in range(len(compiled_eval_data)):
			pt_gt_data.append(compiled_eval_data[meal_num][3])
		# Convert remaining Data to Windows
		for meal_num in range(len(compiled_eval_data)):
			compiled_eval_data[meal_num][3] = TimePoint2Window(np.array(compiled_eval_data[meal_num][3])) # default is 2.5 seconds per bite, centered
	# end of if Converting GT to Windows
	print("Using Clemson Data...") 
else:
	print("Invalid Database flag selected.")
#end of Database switch statement

print("Finished Generating Eval Data.")

Using OREBA Data...
Finished Generating Eval Data.


In [6]:
# # # # # Load Network Model # # # # #
model = tf.keras.models.load_model(MODEL_FILE)

print("Successfully loaded Model.")

2025-08-08 09:20:58.156262: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /software/slurm/spackages/linux-rocky8-x86_64/gcc-12.2.0/anaconda3-2023.09-0-3mhml42fa64byxqyd5fig5tbih625dp2/lib
2025-08-08 09:20:58.156296: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
2025-08-08 09:20:58.156317: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (node0355.palmetto.clemson.edu): /proc/driver/nvidia/version does not exist


Successfully loaded Model.


In [7]:
###### GENERATE ALL DETECTIONS ######


All_Detections = []
All_gt_bite_times = []

for curr_meal in range(len(compiled_eval_data)):
# for curr_meal in [55]:

	# Grab current meal data
	[meal_id,features_input,timesteps,currGT_times]=compiled_eval_data[curr_meal] 
	# Get prediction probabilities
	predictions = model.predict(features_input)
	if DEBUG_PRINTS==1:
		predictions.tofile('Predictions_Raw'+str(curr_meal)+'_subj'+str(FOLD_INDEX)+'.csv', sep = ',')
		np.array(timesteps).tofile('timesteps'+str(curr_meal)+'_subj'+str(FOLD_INDEX)+'.csv', sep = ',')
	# End if DEBUG_PRINT
	for i in range(len(predictions)):
		if predictions[i]<BiteDetectThresh:
			predictions[i]=0
	# predictions[predictions<BiteDetectThresh]=0 # floor any predictions below threshold


	if DETECTION_METHOD_FLAG == 1: # First Method: Floor predictions, then find any local maximum (based on its two neighbors), 
		#then detect the largest LM that ensures all LM are either detections or covered up in 2 sec window 

		# Filter out so only local maxima remain
		predictions_LM=[]
		#take care of front edge cases
		if predictions[0]>predictions[1]:
			predictions_LM.append(1)
		else:
			predictions_LM.append(0)

		#iterate over the rest of the data
		for i in range(1,len(predictions)-1):
			#Bias data towards the left most point in the event of equal values
			if predictions[i]>predictions[i-1] and predictions[i]>=predictions[i+1]:
				predictions_LM.append(1)
			else:
				predictions_LM.append(0)
		#take care of back edge cases
		if predictions[-1]!=0 and predictions[-1]>=predictions[-2]:
			predictions_LM.append(1)
		else:
			predictions_LM.append(0)

		predictions_LM=np.array(predictions_LM)


		Detections=[] # initialize as empty list, add detections as they are found
		for i in range(len(predictions_LM)):
			# if it is a local maximum that needs to be either covered or detected
			if predictions_LM[i]==1:
				# find maximum value within duration ahead to see if it will be covered
				running_max_index=i
				#check the duration ahead for a higher local peak
				for j in range(1,round((bite_length_sec*DATA_FREQ/STRIDE)-0.5)):
					if (i+j)>=len(predictions_LM):
						break
					if predictions_LM[i+j]==1:
						if predictions[running_max_index]<predictions[i+j]:
							# erase previous LM since it is "covered up" by new detection placement
							predictions_LM[running_max_index]=0
							# mark new location of running max 
							running_max_index=i+j
						else:
							predictions_LM[i+j]=0 # Cover up the smaller local maxima
				# Add detection, then cover remaining points up to duration seconds ahead
				Detections.append(timesteps[running_max_index]+delayed_offset_sec)
				for j in range(0,round((bite_length_sec*DATA_FREQ/STRIDE)-0.5)):
					if (running_max_index+j)>=len(predictions_LM):
						break
					if predictions_LM[running_max_index+j]==1:
						predictions_LM[running_max_index+j]=0
			# end of if predictions_LM[i]=1
		#end of for i in range(len(predictions_LM)) loop

	elif DETECTION_METHOD_FLAG == 2: # 2nd Attempt: Grab the maximum in each non-floored segment

		### Knobs ###
		SizeOfGapToMergeSegments=5 #number of predictions needing to be a zero to close off a segment

		### Other Variables ###
		FinishedFlag1=0 # used to mark end of loop across all predictions
		rover1=0 # Main iterator marking where in the prediciton file has been fully inspected
		rover2=0 # when main rover finds a segment, rover 2 scouts ahead to find the 
				# first datapoint after the segment (or end of list)
		Detections=[] # initialize as empty list, add detections as they are found

		while (FinishedFlag1!=1):
			if predictions[rover1]==0: #if no detection, move to next bite
				rover1+=1
				if rover1>=len(predictions)-1: #add the "-1" in order to leave space for rover2 to be inserted
					FinishedFlag1=1
			else: #if the prediction is the start of a bite segment
				FinishedFlag2=0
				rover2=rover1+1 #start scout just ahead of current detected start
				while FinishedFlag2!=1:
					if predictions[rover2]==0:
						FinishedFlag2=1 # Begin with assumption one will end loop if this is the end of segment
						# start at one and add up to SizeOfGapToMergeSegments
						for i in range(1,SizeOfGapToMergeSegments+1):
							# set rover to end if the end of the data is reached
							if rover2+i>=len(predictions):
								rover2=len(predictions)-1
								break # leave i loop and end while since Flag is still false
							if predictions[rover2+i]!=0: #if the gap is short and a non-zero bite is found
								rover2=rover2+i # move scout to the non-zero point
								FinishedFlag2=0 # correct assumption since the gap was short
								break # and continue search in while() loop
						# if loop is exited and FinishedFlag2 has not been cleared, then rover2 is at end
					else: # if predictions[rover2]!=0
						rover2+=1 # move rover along if segment is still occuring
						if rover2>=len(predictions): # check boundary for end of data
							rover2=len(predictions)-1
							FinishedFlag2=1 # or use break statement
				# end of while FinishedFlag2!=1


				# Add Detection as the maximum of the segment
				currMax=predictions[rover1] #start with first point as max
				currMaxIndex=rover1
				for rover3 in range(rover1+1,rover2): # iterate over all remaining data in segment
					if predictions[rover3]>=currMax: # biased to later times if a tie
						currMax=predictions[rover3] 
						currMaxIndex=rover3
				Detections.append(timesteps[currMaxIndex]+delayed_offset_sec)

				# Shift Rover to end of Segment and continue
				rover1=rover2
				if rover1>=len(predictions)-1: #set to minus one from end so that rover2=rover1+1 won't exceed index
					FinishedFlag1=1 #could also just use a break statement

		# end of while FinishedFlag==0
	# end of predictions[] --> Detections[] if statement




	# Convert to np-array to use Eval function 
	Detections = np.array(Detections)
	gt_bite_times = np.array(currGT_times)
	
	
	
	All_Detections.append(Detections)
	All_gt_bite_times.append(gt_bite_times)
	
	print("Through meal {} of {}...".format(curr_meal, len(compiled_eval_data)))


	if DEBUG_PRINTS == 1:
		print("Through meal {} of {}...".format(curr_meal, len(compiled_eval_data)))
	#end of DEBUG_PRINTS
# end of for curr meal loop

print("Finished Producing Detections")

256/256 [==============================] - 20s 75ms/step
Through meal 0 of 20...
188/188 [==============================] - 14s 74ms/step
Through meal 1 of 20...
170/170 [==============================] - 13s 75ms/step
Through meal 2 of 20...
277/277 [==============================] - 20s 74ms/step
Through meal 3 of 20...
277/277 [==============================] - 21s 74ms/step
Through meal 4 of 20...
240/240 [==============================] - 18s 73ms/step
Through meal 5 of 20...
206/206 [==============================] - 15s 73ms/step
Through meal 6 of 20...
283/283 [==============================] - 21s 74ms/step
Through meal 7 of 20...
159/159 [==============================] - 12s 73ms/step
Through meal 8 of 20...
173/173 [==============================] - 13s 73ms/step
Through meal 9 of 20...
233/233 [==============================] - 17s 73ms/step
Through meal 10 of 20...
81/81 [==============================] - 6s 73ms/step
Through meal 11 of 20...
53/53 [======================

In [8]:




# Lists for the metrics of each meal evaluated
All_TP=[] # List (one entry per file) of bites that trigger within ground truth
All_FP=[] # List of Bites that trigger but have already been mapped to a TP
All_FN=[] # List of GT windows without a bite detected

# If using Point vs Window Eval, use these two lists
All_FP1=[] # List of Bites that trigger inside of another TP
All_FP2=[] # List of Bites that trigger outside of any GT


for curr_meal in range(len(compiled_eval_data)):
	
	# Convert to np-array to use Eval function 
	Detections = All_Detections[curr_meal]
	gt_bite_times = All_gt_bite_times[curr_meal] 
	curr_pt_gt_data = pt_gt_data[curr_meal]

	if (len(Detections)==0):
		print("Zero Detections for meal index {}".format(curr_meal))
		print("len = {}, shape = ".format(len(gt_bite_times)),end='')
		print(np.shape(gt_bite_times))
		TP = 0
		FP = 0
		FN = len(gt_bite_times)
	elif EVAL_METHOD_FLAG == 1:
		[TP, FP, FN, Key] = DongEval_PtVsPt(
			Detections, curr_pt_gt_data
		)
	elif EVAL_METHOD_FLAG == 2:
		# GT Should have already be converted to Windows
		[TP, FP, FN, FP1, FP2, Key] = KyritEval_PtVsWindow(
			Detections, gt_bite_times, WIN_TOLERANCE = WIN_TOLERANCE
		)
		All_FP1.append(FP1)
		All_FP2.append(FP2)
	elif EVAL_METHOD_FLAG == 3:
		# Convert Predictions to windows
		Detections_wind = TimePoint2Window(Detections)
		# GT Should have already be converted to Windows
		[TP, FP, FN, TN, Key] = Duration_WindowVsWindow(
			Detections_wind, gt_bite_times
		)
	else:
		print("ERROR: UNKNOWN EVAL METHOD REQUESTED. Please pass in 1 for Dong_PtVSPt, 2 for Kyritsis_PtVsWindow, and 3 for Duration_WindVsWind.")
		raise Exception("ERROR: UNKNOWN EVAL METHOD REQUESTED. Please pass in 1 for Dong_PtVSPt, 2 for Kyritsis_PtVsWindow, and 3 for Duration_WindVsWind.")
	#end of EVAL_METHOD_FLAG Switch


	All_TP.append(TP)
	All_FP.append(FP)
	All_FN.append(FN)
# end of for curr meal loop

print("Finished Scoring Evaluating")

Finished Scoring Evaluating


In [9]:
### PRINT SUMMARY STATEMENTS ###

TP = sum(All_TP) # overwrite the short names with the total results
FP = sum(All_FP)
FN = sum(All_FN)

if EVAL_METHOD_FLAG == 2:
	FP1 = sum(All_FP1)
	FP2 = sum(All_FP2)
	#Verbose flag set to 0
	print("Point2Window Kyritsis Eval WinTol = {}".format(WIN_TOLERANCE))
	PrintStats_SingleLine(TP, FP, FN, FP1 = FP1, FP2 = FP2)
else:
	print("Pt2Pt Dong Eval")
	PrintStats_SingleLine(TP, FP, FN)
# End of if EVAL_METHOD_FLAG

if DEBUG_PRINTS == 1:
	print("FINISHED EVALUATING")
#end DEBUG_PRINTS

Pt2Pt Dong Eval
88.421 87.370 89.498 588    69     85    


In [10]:

##################################
####  MATCH DETECTIONS TO GT  ####
##################################

WIN_TOLERANCE = 8
EVAL_METHOD_FLAG = 2
	# 1 = Dong Eval (Pt vs Pt)
	# 2 = Kyritsis (Pt vs Window)
	# 3 = Duration (Window vs Window)


# Lists for the metrics of each meal evaluated
All_TP=[] # List (one entry per file) of bites that trigger within ground truth
All_FP=[] # List of Bites that trigger but have already been mapped to a TP
All_FN=[] # List of GT windows without a bite detected

# If using Point vs Window Eval, use these two lists
All_FP1=[] # List of Bites that trigger inside of another TP
All_FP2=[] # List of Bites that trigger outside of any GT


for curr_meal in range(len(compiled_eval_data)):
	
	# Convert to np-array to use Eval function 
	Detections = All_Detections[curr_meal]
	gt_bite_times = All_gt_bite_times[curr_meal] 
	curr_pt_gt_data = pt_gt_data[curr_meal]

	if (len(Detections)==0):
		print("Zero Detections for meal index {}".format(curr_meal))
		print("len = {}, shape = ".format(len(gt_bite_times)),end='')
		print(np.shape(gt_bite_times))
		TP = 0
		FP = 0
		FN = len(gt_bite_times)
	elif EVAL_METHOD_FLAG == 1:
		[TP, FP, FN, Key] = DongEval_PtVsPt(
			Detections, curr_pt_gt_data
		)
	elif EVAL_METHOD_FLAG == 2:
		# GT Should have already be converted to Windows
		[TP, FP, FN, FP1, FP2, Key] = KyritEval_PtVsWindow(
			Detections, gt_bite_times,WIN_TOLERANCE = WIN_TOLERANCE
		)
		All_FP1.append(FP1)
		All_FP2.append(FP2)
	elif EVAL_METHOD_FLAG == 3:
		# Convert Predictions to windows
		Detections_wind = TimePoint2Window(Detections)
		# GT Should have already be converted to Windows
		[TP, FP, FN, TN, Key] = Duration_WindowVsWindow(
			Detections_wind, gt_bite_times
		)
	else:
		print("ERROR: UNKNOWN EVAL METHOD REQUESTED. Please pass in 1 for Dong_PtVSPt, 2 for Kyritsis_PtVsWindow, and 3 for Duration_WindVsWind.")
		raise Exception("ERROR: UNKNOWN EVAL METHOD REQUESTED. Please pass in 1 for Dong_PtVSPt, 2 for Kyritsis_PtVsWindow, and 3 for Duration_WindVsWind.")
	#end of EVAL_METHOD_FLAG Switch


	All_TP.append(TP)
	All_FP.append(FP)
	All_FN.append(FN)
# end of for curr meal loop

print("Finished Scoring Evaluating")

Finished Scoring Evaluating


In [11]:
### PRINT SUMMARY STATEMENTS ###

TP = sum(All_TP) # overwrite the short names with the total results
FP = sum(All_FP)
FN = sum(All_FN)

if EVAL_METHOD_FLAG == 2:
	FP1 = sum(All_FP1)
	FP2 = sum(All_FP2)
	#Verbose flag set to 0
	print("Point2Window Kyritsis Eval WinTol = {}".format(WIN_TOLERANCE))
	PrintStats_SingleLine(TP, FP, FN, FP1 = FP1, FP2 = FP2)
else:
	print("Pt2Pt Dong Eval")
	PrintStats_SingleLine(TP, FP, FN)
# End of if EVAL_METHOD_FLAG

if DEBUG_PRINTS == 1:
	print("FINISHED EVALUATING")
#end DEBUG_PRINTS

Point2Window Kyritsis Eval WinTol = 8
83.910 82.912 84.932 558    99     115   
